# 06 Prediction (methods B and C)

Do political facts known on day 0 predict a strong reaction of the national index beyond pre-event volatility,
out of sample?

- **Sample.** Eligible elections and crises after the overlap rule (planned changes are not in it); outcome
  strong at p = `P_MAIN` over day 0 and day +1.
- **Models.** Logistic regression with L2 penalty (ridge), each in a scikit-learn Pipeline(StandardScaler,
  LogisticRegression), so the scaler is fitted on the training data only:
  M0 intercept only (the training base rate), M1 vol_pre, M2 vol_pre + crisis,
  M3 M2 + enp + coalition + minority + lr_shift_el.
- **Outer validation.** Leave one country out; training events whose day 0 is within `PURGE_DAYS` calendar days
  of any event of the test country are removed from that fold's training set. Out-of-fold predictions of all
  folds are pooled before scoring.
- **Inner validation.** C chosen on each training set by GroupKFold by country (`INNER_FOLDS` folds) over
  `C_GRID`, criterion mean log-loss, no class weights.
- **Metrics.** Pooled out-of-fold log-loss (primary), AUC and Brier score. Intervals: country-cluster bootstrap
  of the pooled out-of-fold predictions, no refit (`B_BOOT` replicates, percentile intervals at `CI_LEVEL`).
- **P3** (primary): Δ = log-loss(M1) − log-loss(M3). Null: the block (enp, coalition, minority, lr_shift_el)
  permuted jointly across events within each country × group cell, outcome and vol_pre kept, the nested pipeline
  re-run. `N_PERM` permutations, or `N_PERM_FALLBACK` if the measured time of one run times `N_PERM` exceeds
  `PERM_TIME_LIMIT_MIN` minutes. One-sided p = (1 + #{Δ* ≥ Δ}) / (N + 1). Holm across P1, P2 and P3 is
  computed once, in notebook 08.

Secondary analyses, labelled exploratory and reported in full: S2 at p = `P_SEC`; S4 without Israel; S5 each
robustness set (all three with their metrics and Δ, each with its bootstrap interval; no permutation test);
S8 lasso selection frequencies; S9 M1 on placebo dates; S12 method D (random forest), only if `RUN_D` is set to True. The full-sample ridge fit of
M3 is reported for description only (not a test). Running time of this notebook: about 45 minutes, most of it
the permutations of P3.

**Choices made in code, fixed before any result was looked at**:
- The penalty is passed as `l1_ratio` (0 = ridge with solver lbfgs, 1 = lasso with solver liblinear):
  scikit-learn 1.8 deprecates the `penalty` argument. `max_iter` = 10000.
- Inner validation: GroupKFold without shuffling, on the purged training set; inner folds are not purged.
  On a tie in mean log-loss the smallest C (strongest penalty) wins.
- P3 null: M1 uses vol_pre only, which the permutation keeps, so its out-of-fold predictions are the same in every
  permutation and are computed once; each permutation re-runs the nested pipeline of M3. The time rule uses the
  measured time of the observed nested M3 run, which is the work repeated by each permutation.
- Secondary specifications (S2, S4, S5): Δ with its country-cluster bootstrap interval, no permutation test (a
  permutation null costs about 20 minutes per specification on the course workspace). The interval does not
  refit the models, so it is narrower than a full one: read it as indicative.
- Every resampling procedure (permutations, bootstrap, placebo draws) uses its own numpy Generator seeded with
  `SEED`; every scikit-learn `random_state` is `SEED`.
- Bootstrap: the countries of the sample are resampled (the same draws as in 05); each event is weighted by the
  number of times its country is drawn; AUC is the weighted Mann-Whitney statistic (ties count one half);
  a replicate without a positive or a negative has no AUC and is counted (`n_boot_empty`).
- M0 check: its out-of-fold log-loss must equal, exactly, the entropy of the base rate plus the extra loss of
  predicting each test country with its fold's training base rate; the extra loss is printed by country.
- M0 has no AUC row: its prediction is the training base rate, which is lower exactly when the test country has
  more positives, so its pooled AUC is below 0.5 by construction and carries no information.
- Δ(M0 − M1) is also reported, with its interval, to show how much volatility alone adds over the base rate.
- S9: for every event of the main sample one date d is drawn at random from its matched pool (as in 05) among
  the pool dates whose vol_pre can be computed; vol_pre at d is built exactly as an event's (market model
  fitted on the window placed around d, AR over `PRE_VOL_WIN`, sd over the country's normal-day sd); the
  outcome is the placebo pair's strong flag (|AR(d)| or |AR(d+1)| above the threshold, as for π in 05).
  Folds and purge as for the events, on the placebo dates.

In [1]:
import sys
from pathlib import Path

# On Nuvolos the kernel does not start in the notebook's folder: find code/ as 02_sample does.
roots = [Path.cwd(), Path("/files"), Path.home()]
code_dir = next(p for r in roots for p in [r, *r.glob("**/elections_project/code")] if (p / "helpers.py").exists())
sys.path.insert(0, str(code_dir))
print("Working in", code_dir)

import hashlib

import numpy as np
import pandas as pd
import sklearn
from threadpoolctl import threadpool_limits

from helpers import (C_GRID, N_PERM, N_PERM_FALLBACK, P_MAIN, P_SEC, PERM_TIME_LIMIT_MIN, RF_MIN_LEAF_GRID, SEED,
                     VOL_DDOF, load_derived, load_prices, save_derived)
from analysis import (MODEL_FEATURES, QUANTILES, STRONG, bootstrap_metrics, build_panels, check_folds, day_class,
                      fit_models, inner_choice, load_robustness_sets, logistic_model, market_model,
                      nested_predictions, normal_pairs, outer_folds, p3_test, percentile_interval, placebo_pool,
                      point_metrics, pre_volatility, prediction_sample, prediction_specs, prepare_spec)

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)

assert tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8), \
    "scikit-learn >= 1.8 is needed: with older versions l1_ratio is ignored and the lasso would be a ridge"
# The Nuvolos workspace has one CPU: one thread for the linear algebra avoids oversubscription (results unchanged).
threadpool_limits(limits=1)
print("scikit-learn", sklearn.__version__)

# Switch for method D (random forest, S12). It is turned on only by the decision recorded in a commit message.
RUN_D = False

Working in /files/mywork/elections_project/code
scikit-learn 1.8.0


## Inputs
Events from 03, outcomes and vol_pre from 04. Every sample comes from `apply_robustness_set(events, set,
outcomes)`: events eligible in 03 and 04, overlap rule applied among them; planned changes are left out.

In [2]:
events = load_derived("events", parse_dates=["source_date", "day0", "day1", "eligible_from"])
events["eligible"] = events["eligible"].astype(bool)
events["ineligible_reason"] = events["ineligible_reason"].fillna("")
outcomes = load_derived("event_outcomes")
outcomes["eligible"] = outcomes["eligible"].astype(bool)

assert QUANTILES["q95"] == P_MAIN and QUANTILES["q90"] == P_SEC
Y_MAIN = STRONG["q95"]
FEATURES = MODEL_FEATURES["M3"]

SETS = ["main"] + list(dict.fromkeys(load_robustness_sets()["set"]))
samples = {s: prediction_sample(events, outcomes, s) for s in SETS}

# Counts after eligibility in 03 and 04 and the overlap rule (crises, elections; as in notebook 05)
expected = {"main": (76, 144), "R_add": (83, 144), "R_drop": (74, 144), "R_vol": (70, 144),
            "R_bundle": (78, 144), "R_tv": (60, 144), "R_split": (75, 144)}
counts = {s: tuple(int((t["group"] == g).sum()) for g in ("crisis", "election")) for s, t in samples.items()}
assert counts == expected, counts
for s, t in samples.items():
    assert t[FEATURES + list(STRONG.values())].notna().all().all(), f"{s}: missing feature or outcome"
    assert np.isfinite(t[FEATURES].to_numpy(dtype=float)).all()
    assert (t["crisis"] == (t["group"] == "crisis")).all() and (t.loc[t["crisis"] == 1, "lr_shift_el"] == 0).all()
# inputs are checked finite above, so scikit-learn's own input validation can be skipped (faster, same results)
sklearn.set_config(assume_finite=True, skip_parameter_validation=True)

main = samples["main"]
print(pd.DataFrame(counts, index=["crisis", "election"]).to_string())
print(f"\nMain sample: {len(main)} events in {main['country'].nunique()} countries")
print(main[FEATURES].describe().round(3).to_string())

          main  R_add  R_drop  R_vol  R_bundle  R_tv  R_split
crisis      76     83      74     70        78    60       75
election   144    144     144    144       144   144      144

Main sample: 220 events in 23 countries
       vol_pre   crisis      enp  coalition  minority  lr_shift_el
count  220.000  220.000  220.000    220.000   220.000      220.000
mean    -0.164    0.345    4.275      0.718     0.309        0.205
std      0.429    0.477    1.874      0.451     0.463        0.296
min     -1.246    0.000    1.850      0.000     0.000        0.000
25%     -0.476    0.000    2.712      0.000     0.000        0.000
50%     -0.229    0.000    3.552      1.000     0.000        0.084
75%      0.126    1.000    5.539      1.000     1.000        0.311
max      1.236    1.000    9.698      1.000     1.000        1.908


## Main specification and timing of one run
Positives per fold are printed and not acted on. The M0 line checks M0 exactly: its log-loss must equal the
entropy of the base rate plus the extra loss of predicting each test country with its fold's training base rate.

In [3]:
SPECS = prediction_specs(SETS)
assert SPECS[0] == ("main", "main", Y_MAIN, True)
spec = {}
spec[("main", "main")] = d_main = fit_models(prepare_spec(main, Y_MAIN, True, verbose=True))
print("\nC chosen per fold:")
print(pd.DataFrame(d_main["chosen"], index=[f["country"] for f in d_main["folds"]]).T.to_string())

t_run = d_main["seconds_M3"]
n_perm_main = N_PERM if t_run * N_PERM / 60 <= PERM_TIME_LIMIT_MIN else N_PERM_FALLBACK
print(f"\nOne nested M3 run (the work repeated by each permutation): {t_run:.2f} s; "
      f"x N_PERM = {t_run * N_PERM / 60:.1f} min (limit {PERM_TIME_LIMIT_MIN} min) -> {n_perm_main} permutations, "
      f"about {t_run * n_perm_main / 60:.0f} min")

country  test_events  test_positives  train_events  train_positives  purged
    AUS           13               3           200               34       7
    AUT           12               2           201               36       7
    BEL            8               0           208               38       4
    CAN           10               1           206               38       4
    CHE            6               0           212               39       2
    DEU            7               1           205               37       8
    DNK            2               0           217               39       1
    ESP            9               3           207               35       4
    EST            5               0           214               39       1
    FIN            6               0           209               39       5
    FRA           10               1           203               36       7
    GBR           12               2           204               35       4
    GRC     

## P3: permutation null of the political block (main specification)

In [4]:
p3_test(d_main, n_perm_main)
print(f"P3: Δ = {d_main['delta']:.5f}, p = {d_main['p3']:.4f} ({n_perm_main} permutations)")

  100/999 permutations, 4.4 min
  200/999 permutations, 8.9 min
  300/999 permutations, 13.2 min
  400/999 permutations, 17.7 min
  500/999 permutations, 22.1 min
  600/999 permutations, 26.6 min
  800/999 permutations, 35.5 min
  900/999 permutations, 40.0 min
P3: Δ = -0.00544, p = 0.1030 (999 permutations)


## Secondary specifications: S2 (p = `P_SEC`), S4 (without Israel), S5 (robustness sets)
Each is re-run in full (folds, nested pipeline of M0–M3); metrics, Δ and intervals are computed below.

In [5]:
for variant, set_name, ycol, israel in SPECS[1:]:
    print(f"\n{variant} / {set_name}")
    spec[(variant, set_name)] = fit_models(prepare_spec(samples[set_name], ycol, israel))


q90 / main
23 folds, 70 positives of 220; folds with zero positives in the test country (5, still scored in the pooled metrics): ['BEL', 'CHE', 'DNK', 'LTU', 'SWE']
M0 log-loss 0.6358 = base-rate entropy 0.6255 + 0.0103 from differences in the training base rate across folds (largest: {'TUR': 0.004, 'ITA': 0.0012, 'GRC': 0.001})

no_israel / main
22 folds, 34 positives of 195; folds with zero positives in the test country (8, still scored in the pooled metrics): ['BEL', 'CHE', 'DNK', 'EST', 'FIN', 'LTU', 'NZL', 'SWE']
M0 log-loss 0.4785 = base-rate entropy 0.4627 + 0.0157 from differences in the training base rate across folds (largest: {'TUR': 0.0079, 'GRC': 0.0025, 'JPN': 0.0013})

main / R_add
23 folds, 40 positives of 227; folds with zero positives in the test country (8, still scored in the pooled metrics): ['BEL', 'CHE', 'DNK', 'EST', 'FIN', 'LTU', 'NZL', 'SWE']
M0 log-loss 0.4786 = base-rate entropy 0.4656 + 0.0130 from differences in the training base rate across folds (larges

## S9: M1 on placebo dates
The market-model fits of every day are recomputed with the functions of 04; vol_pre recomputed at the events'
own day 0 must equal the value saved by 04.

In [6]:
political_dates = load_derived("political_dates", parse_dates=["date", "trading_day"])
normal = load_derived("normal_days", parse_dates=["date"])
normal["normal"] = normal["normal"].astype(bool)
thr = load_derived("thresholds")
panels = build_panels(load_prices(), political_dates)
fits = {c: market_model(p) for c, p in panels.items()}
sd_c = normal[normal["normal"]].groupby("country")["ar"].std(ddof=VOL_DDOF)
pairs = normal_pairs(normal, thr)

check = np.array([pre_volatility(panels[c], fits[c], [panels[c].index.get_loc(d0)], sd_c[c])[0]
                  for c, d0 in zip(main["country"], main["day0"])])
assert np.allclose(check, main["vol_pre"], rtol=0, atol=1e-9), "vol_pre does not reproduce 04"

rng = np.random.default_rng(SEED)
rows = []
for e in main.itertuples(index=False):
    pool, radius = placebo_pool(pairs, e.country, e.day0, day_class(e.country, [e.day0])[0])
    pos = panels[e.country].index.get_indexer(pool["date"])
    assert (pos >= 0).all()
    vp = pre_volatility(panels[e.country], fits[e.country], pos, sd_c[e.country])
    valid = np.flatnonzero(np.isfinite(vp))
    assert len(valid), f"{e.event_id}: no placebo date with a computable vol_pre"
    k = rng.choice(valid)
    rows.append({"event_id": e.event_id, "event_variant": e.variant, "country": e.country, "group": e.group,
                 "day0": pool["date"].iloc[k], "vol_pre": vp[k], "pool_valid": len(valid), "pool_pairs": len(pool),
                 Y_MAIN: int(pool["above_q95_0"].iloc[k] or pool["above_q95_1"].iloc[k])})
plac = pd.DataFrame(rows)
print(f"Placebo sample: {len(plac)} dates, {plac[Y_MAIN].sum()} strong ({plac[Y_MAIN].mean():.3f}); "
      f"pool dates without a computable vol_pre: {(plac['pool_pairs'] - plac['pool_valid']).sum()}")

d_plac = {"t": plac, "y": plac[Y_MAIN].to_numpy(), "countries": plac["country"].to_numpy(),
          "X": {"M1": plac[["vol_pre"]].to_numpy(dtype=float)}}
d_plac["folds"] = outer_folds(d_plac["countries"], plac["day0"])
check_folds(d_plac, verbose=False)
d_plac["preds"] = {"M0": nested_predictions(d_plac["X"]["M1"], d_plac["y"], d_plac["countries"], d_plac["folds"], "M0")[0],
                   "M1": nested_predictions(d_plac["X"]["M1"], d_plac["y"], d_plac["countries"], d_plac["folds"], "l2")[0]}
assert all(((p > 0) & (p < 1)).all() for p in d_plac["preds"].values())

Placebo sample: 220 dates, 22 strong (0.100); pool dates without a computable vol_pre: 0
23 folds, 22 positives of 220; folds with zero positives in the test country (5, still scored in the pooled metrics): ['AUS', 'CAN', 'DNK', 'FIN', 'LTU']


## S8: lasso selection frequencies (M3, main specification)
Lasso (l1, solver liblinear), same grid and folds; how often each feature has a nonzero coefficient across the
outer folds. No headline number.

In [7]:
_, lasso_info = nested_predictions(d_main["X"]["M3"], d_main["y"], d_main["countries"], d_main["folds"], "l1")
sel = np.array([i["nonzero"] for i in lasso_info])
lasso = pd.DataFrame({"feature": FEATURES, "n_folds_selected": sel.sum(axis=0), "n_folds": len(sel),
                      "share": sel.mean(axis=0)})
print(lasso.to_string(index=False))
print("C chosen per fold:", [round(float(i["value"]), 4) for i in lasso_info])

    feature  n_folds_selected  n_folds    share
    vol_pre                23       23 1.000000
     crisis                 3       23 0.130435
        enp                20       23 0.869565
  coalition                20       23 0.869565
   minority                 1       23 0.043478
lr_shift_el                23       23 1.000000
C chosen per fold: [1.0, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162, 0.3162]


## Full-sample ridge fit of M3 (description only, not a test)
C chosen by the inner validation on the full main sample. Coefficients are on the standardised scale (per sd of
the feature) and per unit of the feature.

In [8]:
X3, y3 = d_main["X"]["M3"], d_main["y"]
C_full, loss_full = inner_choice(X3, y3, d_main["countries"], lambda C: logistic_model(C, "l2"), C_GRID)
full = logistic_model(C_full, "l2").fit(X3, y3)
lr, sc = full.named_steps["logit"], full.named_steps["scale"]
coefs = pd.DataFrame({"feature": ["intercept"] + FEATURES,
                      "coef_std": np.r_[lr.intercept_, lr.coef_[0]],
                      "coef_per_unit": np.r_[np.nan, lr.coef_[0] / sc.scale_],
                      "feature_mean": np.r_[np.nan, sc.mean_], "feature_sd": np.r_[np.nan, sc.scale_],
                      "C": C_full})
print(coefs.round(4).to_string(index=False))
print("Mean inner log-loss by C:", {round(float(c), 4): round(float(v), 4) for c, v in zip(C_GRID, loss_full)})

    feature  coef_std  coef_per_unit  feature_mean  feature_sd      C
  intercept   -1.5822            NaN           NaN         NaN 0.0316
    vol_pre    0.3397         0.7945       -0.1639      0.4275 0.0316
     crisis    0.0440         0.0925        0.3455      0.4755 0.0316
        enp   -0.0902        -0.0483        4.2753      1.8696 0.0316
  coalition   -0.0895        -0.1990        0.7182      0.4499 0.0316
   minority    0.0177         0.0382        0.3091      0.4621 0.0316
lr_shift_el    0.1090         0.3688        0.2055      0.2956 0.0316
Mean inner log-loss by C: {0.001: 0.4704, 0.0032: 0.4681, 0.01: 0.4633, 0.0316: 0.4593, 0.1: 0.4629, 0.3162: 0.4706, 1.0: 0.4758, 3.1623: 0.478, 10.0: 0.4787, 31.6228: 0.479, 100.0: 0.4791}


## S12: method D (random forest), only if `RUN_D` is True
Random forest on the M3 features, `RF_N_TREES` trees, max_features "sqrt", min_samples_leaf chosen in
`RF_MIN_LEAF_GRID` by the inner validation (same folds, log-loss), `random_state` = `SEED`. Same outer folds,
purge and metrics; main specification with and without Israel. No test, never a headline. A forest can predict
exactly 0 or 1; log-loss then clips the prediction to [eps, 1 − eps], as scikit-learn does.

In [9]:
if RUN_D:
    for key in [("main", "main"), ("no_israel", "main")]:
        d = spec[key]
        d["preds"]["RF"], info = nested_predictions(d["X"]["M3"], d["y"], d["countries"], d["folds"],
                                                    ("rf", RF_MIN_LEAF_GRID))
        print(key, "min_samples_leaf per fold:", [int(i["value"]) for i in info])
else:
    print("Method D not run (RUN_D = False).")

Method D not run (RUN_D = False).


## Results table
One row per specification, model and metric, with the country-cluster bootstrap interval. Only the Δ(M1 − M3)
row of the main specification carries a p-value (P3, primary).

In [10]:
def result_rows(variant, set_name, d, deltas):
    boot = bootstrap_metrics(d["countries"], d["y"], d["preds"], deltas)
    point = {m: point_metrics(d["y"], p) for m, p in d["preds"].items()}
    base = {"variant": variant, "set": set_name, "N": len(d["y"]), "n_pos": int(d["y"].sum())}
    rows = []
    for m in d["preds"]:
        for metric in ("logloss", "auc", "brier"):
            if m == "M0" and metric == "auc":
                continue
            reps = boot[(m, metric)]
            lo, hi = percentile_interval(reps)
            rows.append({**base, "model": m, "metric": metric, "value": point[m][metric], "ci_low": lo, "ci_high": hi,
                         "n_boot_empty": int(np.isnan(reps).sum())})
    for a, b in deltas:
        reps = boot[(f"{a}-{b}", "delta_logloss")]
        lo, hi = percentile_interval(reps)
        row = {**base, "model": f"{a}-{b}", "metric": "delta_logloss",
               "value": point[a]["logloss"] - point[b]["logloss"], "ci_low": lo, "ci_high": hi,
               "n_boot_empty": int(np.isnan(reps).sum())}
        if (a, b) == ("M1", "M3") and "p3" in d:
            assert np.isclose(row["value"], d["delta"], rtol=0, atol=1e-12)
            row.update(p_value=d["p3"], n_perm=d["n_perm"])
        rows.append(row)
    return rows


rows = []
for (variant, set_name), d in spec.items():
    rows += result_rows(variant, set_name, d, [("M0", "M1"), ("M1", "M3")])
rows += result_rows("placebo", "main", d_plac, [("M0", "M1")])

COLS = ["model", "variant", "set", "metric", "value", "ci_low", "ci_high", "n_boot_empty", "N", "n_pos", "p_value",
        "n_perm", "primary"]
res = pd.DataFrame(rows)
res["primary"] = (res["model"] == "M1-M3") & (res["variant"] == "main") & (res["set"] == "main")
res = res.reindex(columns=COLS)
assert res["primary"].sum() == 1

### Primary: P3 (main specification), with the same test without Israel next to it
No multiple-testing correction here: Holm across P1, P2 and P3 is applied in notebook 08.

In [11]:
show = ["model", "variant", "set", "metric", "value", "ci_low", "ci_high", "N", "n_pos", "p_value", "n_perm"]
print(res[(res["model"] == "M1-M3") & (res["set"] == "main") & res["variant"].isin(["main", "no_israel"])][show]
      .round(4).to_string(index=False))
print()
print(res[(res["variant"] == "main") & (res["set"] == "main")][show].round(4).to_string(index=False))

model   variant  set        metric   value  ci_low  ci_high   N  n_pos  p_value  n_perm
M1-M3      main main delta_logloss -0.0054 -0.0197   0.0087 220     39    0.103   999.0
M1-M3 no_israel main delta_logloss -0.0085 -0.0277   0.0150 195     34      NaN     NaN

model variant  set        metric   value  ci_low  ci_high   N  n_pos  p_value  n_perm
   M0    main main       logloss  0.4802  0.3729   0.6012 220     39      NaN     NaN
   M0    main main         brier  0.1495  0.1054   0.1983 220     39      NaN     NaN
   M1    main main       logloss  0.4480  0.3382   0.5628 220     39      NaN     NaN
   M1    main main           auc  0.6344  0.5603   0.7468 220     39      NaN     NaN
   M1    main main         brier  0.1382  0.0963   0.1808 220     39      NaN     NaN
   M2    main main       logloss  0.4540  0.3440   0.5700 220     39      NaN     NaN
   M2    main main           auc  0.6154  0.5449   0.7225 220     39      NaN     NaN
   M2    main main         brier  0.1399  0.097

### Secondary analyses (exploratory)

In [12]:
print(res[~res["primary"]][show].round(4).to_string(index=False))

model   variant      set        metric   value  ci_low  ci_high   N  n_pos  p_value  n_perm
   M0      main     main       logloss  0.4802  0.3729   0.6012 220     39      NaN     NaN
   M0      main     main         brier  0.1495  0.1054   0.1983 220     39      NaN     NaN
   M1      main     main       logloss  0.4480  0.3382   0.5628 220     39      NaN     NaN
   M1      main     main           auc  0.6344  0.5603   0.7468 220     39      NaN     NaN
   M1      main     main         brier  0.1382  0.0963   0.1808 220     39      NaN     NaN
   M2      main     main       logloss  0.4540  0.3440   0.5700 220     39      NaN     NaN
   M2      main     main           auc  0.6154  0.5449   0.7225 220     39      NaN     NaN
   M2      main     main         brier  0.1399  0.0975   0.1832 220     39      NaN     NaN
   M3      main     main       logloss  0.4535  0.3486   0.5608 220     39      NaN     NaN
   M3      main     main           auc  0.5958  0.5070   0.7005 220     39      

### Reproducibility
The nested pipeline of the main specification is re-run; with the same seed the out-of-fold predictions must be
identical. The hash of `res_prediction.csv` printed at the end must be the same on a second run of the notebook.

In [13]:
again = fit_models(prepare_spec(main, Y_MAIN, True))
assert all(np.array_equal(again["preds"][m], d_main["preds"][m]) for m in MODEL_FEATURES.keys() | {"M0"}), \
    "predictions differ"
print("Main-specification predictions identical on a second run.")

23 folds, 39 positives of 220; folds with zero positives in the test country (8, still scored in the pooled metrics): ['BEL', 'CHE', 'DNK', 'EST', 'FIN', 'LTU', 'NZL', 'SWE']
M0 log-loss 0.4802 = base-rate entropy 0.4672 + 0.0130 from differences in the training base rate across folds (largest: {'TUR': 0.0067, 'GRC': 0.0018, 'JPN': 0.001})
Main-specification predictions identical on a second run.


## Outputs
- `oof_predictions.csv`: out-of-fold predictions of every specification (placebo: the drawn date in day0).
- `res_prediction.csv`: metrics with intervals; the primary Δ row with the P3 p-value.
- `perm_null.csv`: the permutation null of Δ of the main specification.
- `res_coefficients.csv`: full-sample ridge fit of M3 (description only).
- `res_lasso_selection.csv`: S8.

In [14]:
oof = []
for (variant, set_name), d in list(spec.items()) + [(("placebo", "main"), d_plac)]:
    t = d["t"]
    o = pd.DataFrame({"variant": variant, "set": set_name, "event_id": t["event_id"],
                      "event_variant": t["event_variant"] if "event_variant" in t else t["variant"],
                      "country": t["country"], "group": t["group"], "day0": t["day0"], "y": d["y"]})
    for m, p in d["preds"].items():
        o[m] = p
    oof.append(o)
oof = pd.concat(oof, ignore_index=True)
perm = pd.DataFrame({"variant": "main", "set": "main", "perm": np.arange(1, n_perm_main + 1), "delta": d_main["null"]})

for nm, df in [("oof_predictions", oof), ("res_prediction", res), ("perm_null", perm),
               ("res_coefficients", coefs), ("res_lasso_selection", lasso)]:
    print(save_derived(df, nm), len(df), "rows")
print("res_prediction hash:", hashlib.sha256(res.to_csv(index=False).encode()).hexdigest())

/files/mywork/elections_project/data/derived/oof_predictions.csv 2159 rows
/files/mywork/elections_project/data/derived/res_prediction.csv 123 rows
/files/mywork/elections_project/data/derived/perm_null.csv 999 rows
/files/mywork/elections_project/data/derived/res_coefficients.csv 7 rows
/files/mywork/elections_project/data/derived/res_lasso_selection.csv 6 rows
res_prediction hash: 9f21e4fb2138dbab1c1c087e9fe9d67d7798948d0b7c8be94a7339db9c0e217e
